In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
!git clone https://github.com/IOAI-official/IOAI-2025.git
%cd IOAI-2025/Individual-Contest/Radar

Cloning into 'IOAI-2025'...
remote: Enumerating objects: 20745, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (6/6), done.
^C
[Errno 2] No such file or directory: 'IOAI-2025/Individual-Contest/Radar'
/content/IOAI-2025/Individual-Contest/Radar


In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class CustomDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)

        images = data[:6]
        labels = data[6]

        images = images.float()
        labels = labels.long()
        labels = labels + 1

        if self.transform:
            images = self.transform(images)
            labels = self.transform(labels)

        return images, labels, self.file_names[idx]

class CustomDataset_test(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)

        images = data[:6]

        images = images.float()

        if self.transform:
            images = self.transform(images)

        return images, self.file_names[idx]



In [ ]:
def generate_file_paths(base_path):
    file_paths = []
    for frame in os.listdir(base_path):
        frame_path = os.path.join(base_path, frame)
        if frame_path.endswith('.mat.pt'):
            file_paths.append(frame_path)
    return [path for path in file_paths if os.path.exists(path)]

def load_data(base_path, batch_size=4, num_workers=2, test_size=0.2):
    file_paths = generate_file_paths(base_path)

    train_paths, test_paths = train_test_split(file_paths, test_size=test_size, random_state=42)

    train_dataset = CustomDataset(file_paths=train_paths)
    test_dataset = CustomDataset(file_paths=test_paths)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        drop_last=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        drop_last=True
    )

    return train_loader, test_loader

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
!pip install tqdm
from tqdm import tqdm

### Model Definition and Training

In [ ]:
class convblock(nn.Module):
    def __init__(self, in_channels, out_channels) -> None:
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.conv_block(x)
        return x

class Encoder(nn.Module):
    def __init__(self, in_channels, out_channels) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.MaxPool2d(2),
            convblock(in_channels, out_channels)
        )

    def forward(self, x):
        x = self.encoder(x)
        return x

class Decoder(nn.Module):
    def __init__(self, in_channels, out_channels) -> None:
        super().__init__()
        self.conv_trans = nn.ConvTranspose2d(in_channels, out_channels, \
            kernel_size=4, stride=2, padding=1)
        self.conv_block = convblock(in_channels, out_channels)

    def forward(self, x1, x2):

        x1 = self.conv_trans(x1)

        x1 = F.interpolate(
            x1,
            size=x2.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        x = torch.cat([x2, x1], dim=1)

        x = self.conv_block(x)

        return x



In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()


        self.block1 = convblock(6, 32)
        self.enc1= Encoder(32, 64)
        self.enc2= Encoder(64, 128)
        self.enc3= Encoder(128, 256)
        self.enc4= Encoder(256, 512)

        self.dec_1 = Decoder(512, 256)
        self.dec_2 = Decoder(256, 128)
        self.dec_3 = Decoder(128, 64)
        self.dec_4 = Decoder(64, 32)
        self.dec_5= Decoder(32, 16)

        self.out_conv = nn.Conv2d(32, 5, 1)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        input_size = x.shape[2:]
        x1 = self.block1(x)
        x2 = self.enc1(x1)
        x3 = self.enc2(x2)
        x4 = self.enc3(x3)
        x5 = self.enc4(x4)

        x = self.dec_1(x5, x4)
        x = self.dec_2(x, x3)
        x = self.dec_3(x, x2)
        x = self.dec_4(x, x1)
        # x= self.dec_5(x, x)

        x = self.dropout(x)
        x=self.out_conv(x)
        x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=False)
        return x

In [ ]:
def train(scheduler, model, train_loader, test_loader, optimizer, criterion, num_epochs=50):
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        batch_count = 0

        for images, labels, _ in tqdm(train_loader):
            images = images.cuda() if torch.cuda.is_available() else images
            labels = labels.cuda() if torch.cuda.is_available() else labels

            outputs = model(images)

            # outputs = F.interpolate(outputs, size=labels.shape[1:], mode='bilinear', align_corners=False)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            batch_count += 1

        avg_train_loss = epoch_loss / batch_count
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        val_batch_count = 0

        with torch.no_grad():
            for images, labels, _ in tqdm(test_loader):
                images = images.cuda() if torch.cuda.is_available() else images
                labels = labels.cuda() if torch.cuda.is_available() else labels

                outputs = model(images)
                # outputs = F.interpolate(outputs, size=labels.shape[1:], mode='bilinear', align_corners=False)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                val_batch_count += 1

        avg_val_loss = val_loss / val_batch_count
        val_losses.append(avg_val_loss)
        scheduler.step(avg_val_loss)
        if (epoch+1) % 2 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], '
                  f'Train Loss: {avg_train_loss:.4f}, '
                  f'Val Loss: {avg_val_loss:.4f}')

    return train_losses, val_losses

In [ ]:
TRAIN_PATH = "./"
# The training set is deployed automatically in the testing machine.
# You notebook can access the TRAIN_PATH even if you do not mount it along with notebook.
data_path = TRAIN_PATH + 'training_set'


train_loader, test_loader = load_data(
    base_path=data_path,
    batch_size=32,
    num_workers=2,
    test_size=0.2
)

model = MyModel()
if torch.cuda.is_available():
    model = model.cuda()

weights = torch.tensor(
    [0.005, 1.0, 1.0, 1.0, 1.0],
    dtype=torch.float32
)

if torch.cuda.is_available():
    weights = weights.cuda()

criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = optim.Adam(model.parameters(), lr=0.0001,weight_decay=0.1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3
    )
train_losses, val_losses = train(
    scheduler=scheduler,
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=50
)

100%|██████████| 11/11 [00:00<00:00, 11.95it/s]


Epoch [2/50], Train Loss: 0.6880, Val Loss: 0.8152


100%|██████████| 11/11 [00:00<00:00, 14.05it/s]


Epoch [4/50], Train Loss: 0.4669, Val Loss: 0.5253


100%|██████████| 11/11 [00:00<00:00, 13.94it/s]


...

Epoch [46/50], Train Loss: 0.1387, Val Loss: 0.1518


100%|██████████| 11/11 [00:00<00:00, 11.92it/s]


Epoch [48/50], Train Loss: 0.1316, Val Loss: 0.1468


100%|██████████| 11/11 [00:00<00:00, 14.14it/s]

Epoch [50/50], Train Loss: 0.1307, Val Loss: 0.1468


In [ ]:
!pip install kaggle

In [ ]:
!mkdir -p ~/.kaggle && echo KGAT_a13222df2cc7984476cf3b3c7bd896bc > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
!kaggle competitions download -c ioai-26-radar

In [ ]:
!unzip ioai-26-radar.zip

### Generate CSV for Submission

In [ ]:
from torch.utils.data import DataLoader
import pandas as pd
import torch
import os

def run_inference(model, data_loader):

    model.eval()

    predictions = []
    filenames = []

    with torch.no_grad():

        for images, file_names in data_loader:

            images = images.cuda() if torch.cuda.is_available() else images

            outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            preds = preds - 1

            for i, pred in enumerate(preds):

                predictions.append(
                    pred.cpu().numpy().flatten()
                )

                filenames.append(file_names[i])

    return predictions, filenames


# ================= TEST =================

TEST_PATH = "/content/IOAI-2025/Individual-Contest/Radar/test_set"

test_paths = generate_file_paths(TEST_PATH)

test_dataset = CustomDataset_test(
    file_paths=test_paths
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

print("Running inference on test set...")

test_predictions, test_filenames = run_inference(
    model,
    test_loader
)

# ================= SAVE =================

results = []

for filename, pred in zip(
    test_filenames,
    test_predictions
):

    row = {"filename": filename}

    for i, p in enumerate(pred):

        row[f"pixel_{i}"] = p

    results.append(row)

submission = pd.DataFrame(results)

submission = submission.sort_values(
    by="filename"
).reset_index(drop=True)

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nSubmission shape:", submission.shape)

print("\nSaved to submission.csv")

Running inference on test set...

Submission shape: (1000, 9051)

Saved to submission.csv
